In [9]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.linear_model import Lasso, Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor
import warnings
import xgboost as xgb
warnings.filterwarnings('ignore')

np.random.seed(42)
sns.set_theme()

In [10]:
train = pd.read_csv("data/track_k_ecommerce_returns_train.csv")
test = pd.read_csv("data/track_k_ecommerce_returns_test.csv")

# Threshold

Our target seems to have a bimodal distribution. <br>
Let's try to find a threshold to separate the features and using different models on them.

In [11]:
from sklearn.tree import DecisionTreeRegressor

X_threshold = train['price_eur']
y = train['return_rate_pct']

threshold_finder = DecisionTreeRegressor(max_depth=1, random_state=42)
threshold_finder.fit(X_threshold.values.reshape(-1, 1), y)

optimal_threshold = threshold_finder.tree_.threshold[0]
print(f"The optimal split point is: {optimal_threshold:.2f}")

The optimal split point is: 173.96


We're using DecisionTreeRegressor at 1 depth to make the sole most efficient cut of our data

In [12]:
df_low_price = train[train['price_eur'] <= optimal_threshold]
df_high_price = train[train['price_eur'] > optimal_threshold]

X_low = df_low_price.drop(columns=['return_rate_pct'])
y_low = df_low_price['return_rate_pct']

X_high = df_high_price.drop(columns=['return_rate_pct'])
y_high = df_high_price['return_rate_pct']

## Test on RandomForest

Separating the datas into two frames so we can train two models.

In [13]:
#Test on RdForest

model_low_tier = RandomForestRegressor(n_estimators=100, random_state=42)
model_high_tier = RandomForestRegressor(n_estimators=100, random_state=42)

model_low_tier.fit(X_low, y_low)
print("Low_Done")
model_high_tier.fit(X_high, y_high)
print("High_Done")

Low_Done
High_Done


In [ ]:
#To hold onto the predictions
predictions = np.zeros(len(test))

#Sort of boolean list that serve as a filter
is_low = test['price_eur'] <= optimal_threshold #[True False True ...] True for low tier, False for high tier
is_high = test['price_eur'] > optimal_threshold #[True False True ...] True for high tier, False for low tier

#Low tier predictions
if is_low.any():
    #New list with index of the low tiers + their features
    X_new_low = test.loc[is_low, model_low_tier.feature_names_in_]

    #Predictions[is_low] targets the index of the low tiers from above
        # [True False True ...] => Index 1 and 3
    predictions[is_low] = model_low_tier.predict(X_new_low)
    
# High tier predictions
if is_high.any():
    #Same principle as low tier, but for high tier

    X_new_high = test.loc[is_high, model_high_tier.feature_names_in_]
    predictions[is_high] = model_high_tier.predict(X_new_high)

#print mse
final_mse = mean_squared_error(test['return_rate_pct'], predictions)
print(f"Final MSE on test set: {final_mse:.4f}")


Final MSE on test set: 25.4354


We have a better MSE than previously using RdForest(28.57) <br>
Now let's try to apply that to the previous Nested CV

## Threshold on Nested CV

In [ ]:
# Here we are going to compare Ridge, Lasso and ElasticNet, each with its own set of hyperparameters. 
# We will use nested cross-validation to do that, which allows us to get an unbiased estimate of the test performance of the best model.

from xgboost import XGBRegressor as XGBoost


# Gradient Boost
from sklearn.ensemble import GradientBoostingRegressor

#Adaboost
from sklearn.ensemble import AdaBoostRegressor

#SVM
from sklearn import svm

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression

outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)

param_ridge = {"alpha": np.logspace(-2, 4, 20)}
param_lasso = {"alpha": np.logspace(-4, 1, 20)}
param_elasticnet = {"alpha": np.logspace(-4, 1, 20), "l1_ratio": np.linspace(0, 1, 20)}
param_randomforest = {"n_estimators": [50, 100], "max_depth": [None, 5, 10]}

#NOTE: AI GENERATED PARAMS
param_xgboost = {
    "n_estimators": [50, 100],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.1, 0.2]
}

param_gradientboost = {
    "n_estimators": [50, 100],
    "max_depth": [3, 5, 7], 
    "learning_rate": [0.01, 0.1, 0.2]
}

param_adaboost = {
    "n_estimators": [50, 100],
    "learning_rate": [0.01, 0.1, 0.2]
}

param_polynomials = {
    'poly__degree': [i for i in range(2, 5)] # This will test linear (1), quadratic (2), and cubic (3)
}

models = {
    'ridge': Ridge(),
    'lasso': Lasso(),
    'elasticnet': ElasticNet(),
    'randomforest': RandomForestRegressor(random_state=42),
    'xgboost': XGBoost(),
    'gradientboost': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'polynomial': Pipeline([
    ('poly', PolynomialFeatures(degree=4)), 
    ('scaler', StandardScaler()),
    ('linear', LinearRegression())
])

}

param_grids = {
    'ridge': param_ridge,
    'lasso': param_lasso,
    'elasticnet': param_elasticnet,
    'randomforest': param_randomforest,
    'xgboost': param_xgboost,
    'gradientboost': param_gradientboost,
    'adaboost': param_adaboost,
    'polynomial': param_polynomials
}


X = train.drop(columns=['return_rate_pct'])
y = train['return_rate_pct']

In [16]:
outer_scores = {name: [] for name in models.keys()}
optimal_threshold = threshold_finder.tree_.threshold[0]

for train_idx, val_idx in outer_cv.split(X, y):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    train_low_mask = X_train['price_eur'] <= optimal_threshold
    val_low_mask = X_val['price_eur'] <= optimal_threshold
    
    X_train_low, y_train_low = X_train[train_low_mask], y_train[train_low_mask]
    X_train_high, y_train_high = X_train[~train_low_mask], y_train[~train_low_mask]
    
    X_val_low, y_val_low = X_val[val_low_mask], y_val[val_low_mask]
    X_val_high, y_val_high = X_val[~val_low_mask], y_val[~val_low_mask]

    inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)
    for name, model in models.items():
        # Tune on Low segment
        grid_low = GridSearchCV(model, param_grids[name], cv=inner_cv)
        grid_low.fit(X_train_low, y_train_low)
        
        # Tune on High segment
        grid_high = GridSearchCV(model, param_grids[name], cv=inner_cv)
        grid_high.fit(X_train_high, y_train_high)
        
        preds = np.zeros(len(X_val))
        
        # Predict using tuned models
        if len(X_val_low) > 0:
            preds[val_low_mask] = grid_low.best_estimator_.predict(X_val_low)
        if len(X_val_high) > 0:
            preds[~val_low_mask] = grid_high.best_estimator_.predict(X_val_high)
            
        fold_mse = mean_squared_error(y_val, preds)
        outer_scores[name].append(fold_mse)

for name, scores in outer_scores.items():
    print(f"{name} | Avg MSE: {np.mean(scores):.4f} | Std: {np.std(scores):.4f}")



ridge | Avg MSE: 57.1191 | Std: 1.8456
lasso | Avg MSE: 57.1578 | Std: 1.6153
elasticnet | Avg MSE: 57.3180 | Std: 1.9145
randomforest | Avg MSE: 28.3466 | Std: 1.8956
xgboost | Avg MSE: 26.1514 | Std: 1.7260
gradientboost | Avg MSE: 26.3381 | Std: 1.9784
adaboost | Avg MSE: 32.1543 | Std: 2.4434
polynomial | Avg MSE: 29.2318 | Std: 2.3206


Unfortunately the threshold separation seems to make the prediction even worse.

## Different Model Based on the Threshold

We did the Nested CV using the same two model for both the high and low ones. <br>
Instead, we'll try to find the best model for each part of the datas and try that instead.

## LOW SEGMENT

In [ ]:
low_segment_results = {name: [] for name in models.keys()}

print("--- Evaluating Models for the LOW TIER ---")

for train_idx, val_idx in outer_cv.split(X, y):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    train_low_mask = X_train['price_eur'] <= optimal_threshold
    val_low_mask = X_val['price_eur'] <= optimal_threshold
    
    X_train_low, y_train_low = X_train[train_low_mask], y_train[train_low_mask]
    X_val_low, y_val_low = X_val[val_low_mask], y_val[val_low_mask]
    
    if len(X_train_low) < 5 or len(X_val_low) == 0: #Skip if not enough
        continue

    inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)
    
    for name, model in models.items():
        grid = GridSearchCV(model, param_grids[name], cv=inner_cv, scoring='neg_mean_squared_error')
        grid.fit(X_train_low, y_train_low)
        
        preds = grid.best_estimator_.predict(X_val_low)
        mse = mean_squared_error(y_val_low, preds)
        low_segment_results[name].append(mse)
        
sort = sorted(low_segment_results.items(), key=lambda x: np.mean(x[1]))

print("\nResults for Low Tier:")
for name, scores in sort:
    print(f"{name.upper()} | Avg MSE: {np.mean(scores):.4f} | Std: {np.std(scores):.4f}")

--- Evaluating Models for the LOW TIER ---
Completed fold with 156 LOW samples.
Completed fold with 154 LOW samples.
Completed fold with 168 LOW samples.
Completed fold with 161 LOW samples.
Completed fold with 157 LOW samples.

Results for Low Tier:
POLYNOMIAL | Avg MSE: 11.1262 | Std: 1.3777
GRADIENTBOOST | Avg MSE: 12.9350 | Std: 1.2293
XGBOOST | Avg MSE: 12.9585 | Std: 1.1199
RANDOMFOREST | Avg MSE: 16.9490 | Std: 0.5331
ADABOOST | Avg MSE: 22.3391 | Std: 1.6029
LASSO | Avg MSE: 47.5869 | Std: 2.8103
RIDGE | Avg MSE: 47.5927 | Std: 2.8747
ELASTICNET | Avg MSE: 47.6357 | Std: 2.9108


The best Model for the low ones seems to be Polynomial at 11.12 

## HIGH

In [18]:
high_segment_results = {name: [] for name in models.keys()}

print("--- Evaluating Models for the HIGH TIER ---")

for train_idx, val_idx in outer_cv.split(X, y):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    train_high_mask = X_train['price_eur'] > optimal_threshold
    val_high_mask = X_val['price_eur'] > optimal_threshold
    
    X_train_high, y_train_high = X_train[train_high_mask], y_train[train_high_mask]
    X_val_high, y_val_high = X_val[val_high_mask], y_val[val_high_mask]
    
    if len(X_train_high) < 5 or len(X_val_high) == 0:
        continue

    inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)
    
    for name, model in models.items():
        grid = GridSearchCV(model, param_grids[name], cv=inner_cv, scoring='neg_mean_squared_error')
        grid.fit(X_train_high, y_train_high)
        
        preds = grid.best_estimator_.predict(X_val_high)
        mse = mean_squared_error(y_val_high, preds)
        high_segment_results[name].append(mse)

sort = sorted(high_segment_results.items(), key=lambda x: np.mean(x[1]))

print("\nResults for High Tier:")
for name, scores in sort:
    print(f"{name.upper()} | Avg MSE: {np.mean(scores):.4f} | Std: {np.std(scores):.4f}")

--- Evaluating Models for the HIGH TIER ---

Results for High Tier:
RANDOMFOREST | Avg MSE: 46.1363 | Std: 3.8940
ADABOOST | Avg MSE: 46.5693 | Std: 5.0424
XGBOOST | Avg MSE: 46.8142 | Std: 3.7720
GRADIENTBOOST | Avg MSE: 47.3495 | Std: 4.1770
POLYNOMIAL | Avg MSE: 57.8010 | Std: 7.4382
LASSO | Avg MSE: 72.2456 | Std: 2.5822
ELASTICNET | Avg MSE: 72.4615 | Std: 2.7451
RIDGE | Avg MSE: 72.5465 | Std: 3.1780


The best Model for the high ones seems to be RandomForest at 46.13.

We'll use a function to test different pairs of models with Nested CV to see which performs the best.

In [ ]:
def Bimodal_model_test(X, y, low_model, high_model):
    bimodal_mse_scores = []

    for train_idx, val_idx in outer_cv.split(X,y):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        train_low_mask = X_train['price_eur'] <= optimal_threshold
        val_low_mask = X_val['price_eur'] <= optimal_threshold

        preds = np.zeros(len(X_val))

        #low
        if train_low_mask.sum() > 5:
            grid_low = GridSearchCV(models[low_model], param_grids[low_model], cv=3)
            grid_low.fit(X_train[train_low_mask], y_train[train_low_mask])
            
            if val_low_mask.any():
                preds[val_low_mask] = grid_low.best_estimator_.predict(X_val[val_low_mask])
        
        #high
        if (~train_low_mask).sum() > 5:
            grid_high = GridSearchCV(models[high_model], param_grids[high_model], cv=3)
            grid_high.fit(X_train[~train_low_mask], y_train[~train_low_mask])
            
            if (~val_low_mask).any():
                preds[~val_low_mask] = grid_high.best_estimator_.predict(X_val[~val_low_mask])
                   
        fold_mse = mean_squared_error(y_val, preds)
        bimodal_mse_scores.append(fold_mse)

    print(f"Bimodal Strategy ({low_model}-Low / {high_model}-High) MSE: {np.mean(bimodal_mse_scores):.4f} (± {np.std(bimodal_mse_scores):.4f})")

# Bimodal Model: Polynomial-Low / RandomForest-High

In [34]:
Bimodal_model_test(X,y,'polynomial', "randomforest")

Bimodal Strategy (polynomial-Low / randomforest-High) MSE: 24.7528 (± 2.2737)


The Bimodal Model using Polynomial Regression for lower features and Random Forest for higher features has the better overall performance.<br>
Focusing on finding a threshold seems to have been the good choice in this situation.

In [ ]:
#OUR MODEL
def Bimodal_PolyLow_RfHigh(train, test):
    optimal_threshold = 173.96  
    model_randomforest = RandomForestRegressor(random_state=42)
    model_poly = Pipeline([
        ('poly', PolynomialFeatures(degree=4)), 
        ('scaler', StandardScaler()),
        ('linear', LinearRegression())
    ])

    param_randomforest = {
        "n_estimators": [50, 100], 
        "max_depth": [None, 5, 10]
    }
    poly_params = {
        'poly__degree': [i for i in range(0, 6)] # This will test linear (1), quadratic (2), and cubic (3)
    }

    #PrepData
    X_train_full = train.drop(columns=['return_rate_pct'])
    y_train_full = train['return_rate_pct']

    X_test = test.drop(columns=['return_rate_pct'])
    y_test_real = test['return_rate_pct']

    train_low_mask = X_train_full['price_eur'] <= optimal_threshold
    test_low_mask = X_test['price_eur'] <= optimal_threshold

    #Training
    grid_low = GridSearchCV(model_poly, poly_params, cv=3)
    grid_low.fit(X_train_full[train_low_mask], y_train_full[train_low_mask])
    
    grid_high = GridSearchCV(model_randomforest, param_randomforest, cv=3)
    grid_high.fit(X_train_full[~train_low_mask], y_train_full[~train_low_mask])

    #Predicting
    final_test_predictions = np.zeros(len(X_test))

    if test_low_mask.any():
        final_test_predictions[test_low_mask] = grid_low.best_estimator_.predict(X_test[test_low_mask])
        
    if (~test_low_mask).any():
        final_test_predictions[~test_low_mask] = grid_high.best_estimator_.predict(X_test[~test_low_mask])


    if y_test_real is not None:
        final_mse = mean_squared_error(y_test_real, final_test_predictions)
        print(f"MSE: {final_mse:.4f}")
    
    return final_test_predictions

# Bimodal Model: GradiengBoost-Low / RandomForest-High

In [35]:
Bimodal_model_test(X,y,'gradientboost', "randomforest")

Bimodal Strategy (gradientboost-Low / randomforest-High) MSE: 25.6283 (± 2.1562)


# Bimodal Model: XGBoost-Low / RandomForest-High


In [36]:
Bimodal_model_test(X,y,'xgboost', "randomforest")

Bimodal Strategy (xgboost-Low / randomforest-High) MSE: 25.8596 (± 2.2648)


Both the Bimodal using GradientBoost and XGBoost are outperformed by the Polynomial.

As the results is lower there's no need to use them; however it is still interesting to look at their performance.